# History Compression 001 — Replay Budget Ladder

Formal hosted-GPU orchestration only. Scientific logic is frozen under `research/validations/history-compression-001/`. Detailed child output is written to local log files; notebook stdout intentionally shows only compact stage summaries.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

ROOT = Path('/kaggle/working/mini-cells')
BRANCH = 'codex/history-compression-001'
if not ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/ArcheLabs/mini-cells.git', str(ROOT)], check=True)
subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=ROOT, check=True)
subprocess.run(['git', 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], cwd=ROOT, check=True)
print({'head': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True).strip()})

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.0.0', 'huggingface_hub==1.11.0', 'safetensors==0.7.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[lm,dev]'], cwd=ROOT, check=True)

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['GITHUB_TOKEN'] = secrets.get_secret('GITHUB_TOKEN')
    try:
        os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
    except Exception:
        pass
except Exception as exc:
    raise RuntimeError('Kaggle Secret GITHUB_TOKEN is required') from exc

subprocess.run([
    sys.executable, 'scripts/research/history_compression_001/publish.py',
    '--branch', BRANCH, '--preflight-only'
], cwd=ROOT, check=True)

In [ ]:
import torch, transformers, huggingface_hub, safetensors
assert torch.cuda.is_available(), 'CUDA is required for the formal run'
protocol_path = ROOT / 'research/validations/history-compression-001/protocol.json'
protocol = json.loads(protocol_path.read_text())
formal_seeds = protocol['formal_seeds']
print({
    'gpu': torch.cuda.get_device_name(0),
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'huggingface_hub': huggingface_hub.__version__,
    'safetensors': safetensors.__version__,
    'formal_seeds': formal_seeds,
    'modes': [(m['id'], m['history_prompt_count']) for m in protocol['compression_modes']],
})

In [ ]:
subprocess.run([
    sys.executable, 'scripts/research/history_compression_001/kaggle_preflight.py',
    '--minimum-free-mb', '12000'
], cwd=ROOT, check=True)

In [ ]:
def run_compact(command, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('w', encoding='utf-8') as handle:
        process = subprocess.Popen(
            command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, env={**os.environ, 'HF_HUB_DISABLE_PROGRESS_BARS': '1', 'TRANSFORMERS_NO_ADVISORY_WARNINGS': '1'}
        )
        assert process.stdout is not None
        for line in process.stdout:
            handle.write(line)
            handle.flush()
            if line.startswith('[hc001]'):
                print(line, end='')
        returncode = process.wait()
    if returncode != 0:
        tail = log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-60:]
        print('\n=== Child log tail ===')
        print('\n'.join(tail))
        subprocess.run(['nvidia-smi'], check=False)
        raise RuntimeError(f'formal child failed with exit code {returncode}; full log: {log_path}')

artifact_root = ROOT / 'artifacts/experiments/history-compression-001'
for seed in formal_seeds:
    durable = artifact_root / f'seed-{seed}/seed_summary.json'
    if durable.is_file():
        print(f'[hc001][seed={seed}] already published; skipping')
        continue
    subprocess.run([
        sys.executable, 'scripts/research/history_compression_001/kaggle_preflight.py',
        '--minimum-free-mb', '12000'
    ], cwd=ROOT, check=True)
    run_compact([
        sys.executable, 'scripts/research/history_compression_001/run_formal_seed.py',
        '--seed', str(seed), '--device', 'cuda:0'
    ], ROOT / f'results/history-compression-001-launcher/seed-{seed}.log')
    subprocess.run([
        sys.executable, 'scripts/research/history_compression_001/publish.py',
        '--seed', str(seed), '--branch', BRANCH
    ], cwd=ROOT, check=True)
    decision = json.loads((artifact_root / 'decision.json').read_text())
    print({
        'status': decision['status'],
        'completed_seeds': decision['completed_seeds'],
        'minimum_observed_supported_history_prompts': decision['minimum_observed_supported_history_prompts'],
        'per_mode': {k: {'supported': v['supported'], 'passed': v['passed_seeds']} for k, v in decision['per_mode'].items()},
    })

In [ ]:
decision_path = ROOT / 'artifacts/experiments/history-compression-001/decision.json'
svg_path = ROOT / 'artifacts/experiments/history-compression-001/visualization/history-compression-summary.svg'
if decision_path.is_file():
    decision = json.loads(decision_path.read_text())
    print(json.dumps({
        'status': decision['status'],
        'scientific_decision': decision['scientific_decision'],
        'minimum_observed_supported_history_prompts': decision['minimum_observed_supported_history_prompts'],
        'support_monotone_with_history_budget': decision['support_monotone_with_history_budget'],
    }, indent=2))
if svg_path.is_file():
    from IPython.display import SVG, display
    display(SVG(filename=str(svg_path)))

Recovery rule: rerun the notebook. Each completed seed is published immediately and skipped on restart. Scientific FAIL modes are durable evidence and are still published. Full child logs remain under `results/history-compression-001-launcher/` and are not streamed wholesale into the notebook page.